# 05 — Construire le dataset final à quatre directions

Ce notebook produit un dataset sans chevauchement de sources dans les proportions demandées : **30 % `bci_bci`**, **28 % `fr_fr`**, **21 % `fr_bci`** et **21 % `bci_fr`**.

Il doit être exécuté après la fin du notebook `04_traduire_extension_bci_fr.ipynb`. Chaque source nettoyée du dataset français est utilisée exactement une fois.

## 1. Configuration et chargement

In [6]:
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import re

PROJECT_ROOT = Path.cwd()
SPLIT_DIR = PROJECT_ROOT / "data" / "01_splits"
FR_BCI_DIR = PROJECT_ROOT / "data" / "02b_text_browser"
BCI_BCI_DIR = PROJECT_ROOT / "data" / "02c_bau_bau_browser"
EXTENSION_DIR = PROJECT_ROOT / "data" / "03_final" / "bci_fr_extension_translation"
FINAL_DIR = PROJECT_ROOT / "data" / "03_final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_DATASET_PATH = FINAL_DIR / "lree_ai_fr_bci_4way_no_overlap_55k.jsonl"
FINAL_MANIFEST_PATH = FINAL_DIR / "lree_ai_fr_bci_4way_no_overlap_55k_manifest.json"
SELECTION_SEED = 20260925
EXTENSION_CANDIDATES = 6_200
TARGET_COUNTS = {"bci_bci": 16_506, "fr_fr": 15_406, "fr_bci": 11_554, "bci_fr": 11_554}
FINAL_ORDER = ("bci_bci", "fr_fr", "fr_bci", "bci_fr")

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_jsonl_atomic(path, records):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    temporary.replace(path)

def write_json_atomic(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

BR_PATTERN = re.compile(r"<br\s*/?>", flags=re.IGNORECASE)
NBSP_PATTERN = re.compile(r"&nbsp;", flags=re.IGNORECASE)
def normalize_text(value):
    text = "" if value is None else str(value)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = BR_PATTERN.sub("\n", text)
    text = NBSP_PATTERN.sub(" ", text).replace("\u00a0", " ")
    text = "\n".join(line.rstrip() for line in text.split("\n"))
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def stable_rank(record, salt):
    value = f"{SELECTION_SEED}|{salt}|{record['id']}"
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

def signature(record):
    return (record["instruction"], record["input"], record["output"])

sources = {
    "bci_pool": sorted(read_jsonl(SPLIT_DIR / "bau_bau_source.jsonl"), key=lambda row: stable_rank(row, "bci_pool")),
    "fr_bci_pool": sorted(read_jsonl(SPLIT_DIR / "fr_bau_source.jsonl"), key=lambda row: stable_rank(row, "fr_bci_pool")),
    "fr_pool": sorted(read_jsonl(SPLIT_DIR / "fr_fr_source.jsonl"), key=lambda row: stable_rank(row, "bci_fr_extension")),
}
assert {key: len(value) for key, value in sources.items()} == {"bci_pool": 22_008, "fr_bci_pool": 22_008, "fr_pool": 11_004}
print("Sortie finale :", FINAL_DATASET_PATH)
print("Ordre         :", " → ".join(FINAL_ORDER))

Sortie finale : /home/vedem_worker/scrapt-bau/data/03_final/lree_ai_fr_bci_4way_no_overlap_55k.jsonl
Ordre         : bci_bci → fr_fr → fr_bci → bci_fr


## 2. Valider et indexer toutes les traductions

In [7]:
report_paths = {
    "fr_bci": FR_BCI_DIR / "validation_report.json",
    "bci_bci": BCI_BCI_DIR / "validation_report.json",
    "bci_fr_extension": EXTENSION_DIR / "validation_report.json",
}
for name, path in report_paths.items():
    if not path.exists():
        raise RuntimeError(f"Rapport absent pour {name}: exécutez d'abord le notebook 04.")
translation_reports = {name: read_json(path) for name, path in report_paths.items()}
for name, report in translation_reports.items():
    assert report["completed_batches"] == report["total_batches"], f"Traduction incomplète : {name}"
    assert report["validation_error_count"] == 0, f"Erreurs de traduction : {name}"

def index_translations(path, expected_direction):
    index = {}
    for record in read_jsonl(path):
        assert record["direction"] == expected_direction
        assert record["complete"] and record["translated_text"] is not None
        key = (record["record_id"], record["target_field"])
        assert key not in index, f"Traduction dupliquée : {key}"
        index[key] = normalize_text(record["translated_text"])
    return index

bci_full = index_translations(BCI_BCI_DIR / "partial_translations.jsonl", "bau_bau")
fr_bci_outputs = index_translations(FR_BCI_DIR / "partial_translations.jsonl", "fr_bau")
bci_fr_extension = index_translations(EXTENSION_DIR / "partial_translations.jsonl", "bci_fr")

extension_sources = sources["fr_pool"][:EXTENSION_CANDIDATES]
expected_extension_keys = set()
for row in extension_sources:
    expected_extension_keys.add((row["id"], "instruction_bci"))
    if normalize_text(row["input_fr"]):
        expected_extension_keys.add((row["id"], "input_bci"))
assert set(bci_fr_extension) == expected_extension_keys
print("Traductions bci complètes :", len(bci_full))
print("Réponses fr_bci          :", len(fr_bci_outputs))
print("Extension bci_fr        :", len(bci_fr_extension))
print("✓ Toutes les traductions sont complètes et alignées")

Traductions bci complètes : 50966
Réponses fr_bci          : 22008
Extension bci_fr        : 8140
✓ Toutes les traductions sont complètes et alignées


## 3. Allouer les sources sans chevauchement et construire les lignes

In [8]:
def make_record(source, direction, source_language, target_language, instruction, input_text, output):
    return {
        "id": f"{source['id']}__{direction}",
        "source_id": source["id"],
        "direction": direction,
        "source_language": source_language,
        "target_language": target_language,
        "source_dataset": source["source_dataset"],
        "source_row_index": source["source_row_index"],
        "instruction": normalize_text(instruction),
        "input": normalize_text(input_text),
        "output": normalize_text(output),
    }

bci_pool = sources["bci_pool"]
bci_bci_rows = []
selected_bci_bci_ids = set()
used_signatures = set()

# Google peut faire converger deux sources françaises différentes vers le même
# triplet baoulé. On sélectionne donc bci_bci par signature unique plutôt que
# par simple tranche, sans jamais réutiliser une source.
for row in bci_pool:
    candidate = make_record(
        row, "bci_bci", "bci", "bci",
        bci_full[(row["id"], "instruction_bau")],
        bci_full.get((row["id"], "input_bau"), ""),
        bci_full[(row["id"], "output_bau")],
    )
    if signature(candidate) in used_signatures:
        continue
    bci_bci_rows.append(candidate)
    selected_bci_bci_ids.add(row["id"])
    used_signatures.add(signature(candidate))
    if len(bci_bci_rows) == TARGET_COUNTS["bci_bci"]:
        break
assert len(bci_bci_rows) == TARGET_COUNTS["bci_bci"]

fr_bci_pool = sources["fr_bci_pool"]
fr_bci_sources = fr_bci_pool[:TARGET_COUNTS["fr_bci"]]
fr_bci_rows = []
for row in fr_bci_sources:
    candidate = make_record(
        row, "fr_bci", "fr", "bci",
        row["instruction_fr"], row["input_fr"], fr_bci_outputs[(row["id"], "output_bau")],
    )
    assert signature(candidate) not in used_signatures, f"Doublon fr_bci inattendu : {row['id']}"
    fr_bci_rows.append(candidate)
    used_signatures.add(signature(candidate))

# Les sources du pool bci non retenues pour bci_bci vont en priorité dans
# bci_fr. Si leur triplet reste identique à un bci_bci, elles passent en fr_fr
# et un candidat traduit de la réserve prend leur place dans bci_fr.
bci_fr_rows = []
reassigned_to_fr_fr = []
for row in bci_pool:
    if row["id"] in selected_bci_bci_ids:
        continue
    candidate = make_record(
        row, "bci_fr", "bci", "fr",
        bci_full[(row["id"], "instruction_bau")],
        bci_full.get((row["id"], "input_bau"), ""),
        row["output_fr"],
    )
    if signature(candidate) in used_signatures:
        reassigned_to_fr_fr.append(row)
        continue
    bci_fr_rows.append(candidate)
    used_signatures.add(signature(candidate))

selected_extension_ids = set()
for row in extension_sources:
    candidate = make_record(
        row, "bci_fr", "bci", "fr",
        bci_fr_extension[(row["id"], "instruction_bci")],
        bci_fr_extension.get((row["id"], "input_bci"), ""),
        row["output_fr"],
    )
    if signature(candidate) in used_signatures:
        continue
    bci_fr_rows.append(candidate)
    selected_extension_ids.add(row["id"])
    used_signatures.add(signature(candidate))
    if len(bci_fr_rows) == TARGET_COUNTS["bci_fr"]:
        break
assert len(bci_fr_rows) == TARGET_COUNTS["bci_fr"], "La réserve bci_fr est insuffisante."

fr_fr_sources = (
    fr_bci_pool[TARGET_COUNTS["fr_bci"]:]
    + [row for row in sources["fr_pool"] if row["id"] not in selected_extension_ids]
    + reassigned_to_fr_fr
)
assert len(fr_fr_sources) == TARGET_COUNTS["fr_fr"]
fr_fr_rows = [
    make_record(row, "fr_fr", "fr", "fr", row["instruction_fr"], row["input_fr"], row["output_fr"])
    for row in fr_fr_sources
]

partition_rows = {
    "bci_bci": bci_bci_rows,
    "fr_fr": fr_fr_rows,
    "fr_bci": fr_bci_rows,
    "bci_fr": bci_fr_rows,
}
final_rows = [row for direction in FINAL_ORDER for row in partition_rows[direction]]
print("Répartition construite :", {key: len(value) for key, value in partition_rows.items()})
print("Sources bci réaffectées à fr_fr :", len(reassigned_to_fr_fr))
print("Candidats de réserve non utilisés :", EXTENSION_CANDIDATES - len(selected_extension_ids))

Répartition construite : {'bci_bci': 16506, 'fr_fr': 15406, 'fr_bci': 11554, 'bci_fr': 11554}
Sources bci réaffectées à fr_fr : 2
Candidats de réserve non utilisés : 146


## 4. Vérifier les proportions, l'unicité et l'absence de chevauchement

In [9]:
expected_total = sum(TARGET_COUNTS.values())
direction_counts = Counter(row["direction"] for row in final_rows)
source_ids = [row["source_id"] for row in final_rows]
row_ids = [row["id"] for row in final_rows]
training_signatures = [signature(row) for row in final_rows]
expected_sequence = [direction for direction in FINAL_ORDER for _ in range(TARGET_COUNTS[direction])]

assert expected_total == 55_020
assert len(final_rows) == expected_total
assert dict(direction_counts) == TARGET_COUNTS
assert [row["direction"] for row in final_rows] == expected_sequence
assert len(set(row_ids)) == expected_total
assert len(set(source_ids)) == expected_total, "Chevauchement de sources détecté."
assert len(set(training_signatures)) == expected_total, "Exemples d'entraînement dupliqués détectés."
assert all(row["instruction"] and row["output"] for row in final_rows)
assert all(row["input"] is not None for row in final_rows)
assert not any(BR_PATTERN.search(row[field]) for row in final_rows for field in ("instruction", "input", "output"))
assert not any(NBSP_PATTERN.search(row[field]) for row in final_rows for field in ("instruction", "input", "output"))

actual_fractions = {direction: direction_counts[direction] / expected_total for direction in FINAL_ORDER}
print("Total                   :", len(final_rows))
print("Sources uniques         :", len(set(source_ids)))
print("Triplets uniques        :", len(set(training_signatures)))
print("Chevauchements          :", len(source_ids) - len(set(source_ids)))
print("Proportions réelles     :", {key: f"{value:.4%}" for key, value in actual_fractions.items()})
print("✓ Dataset sans chevauchement ni doublon")

Total                   : 55020
Sources uniques         : 55020
Triplets uniques        : 55020
Chevauchements          : 0
Proportions réelles     : {'bci_bci': '30.0000%', 'fr_fr': '28.0007%', 'fr_bci': '20.9996%', 'bci_fr': '20.9996%'}
✓ Dataset sans chevauchement ni doublon


## 5. Écrire le dataset final dans `data/03_final`

In [10]:
write_jsonl_atomic(FINAL_DATASET_PATH, final_rows)
output_sha256 = sha256_file(FINAL_DATASET_PATH)
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_name": "Lree AI Labs French–Baoulé Four-Direction Instruction Dataset",
    "output_file": FINAL_DATASET_PATH.name,
    "output_sha256": output_sha256,
    "rows": len(final_rows),
    "ordering": list(FINAL_ORDER),
    "partitions": {direction: {"rows": len(partition_rows[direction]), "fraction": len(partition_rows[direction]) / len(final_rows)} for direction in FINAL_ORDER},
    "source_overlap_count": 0,
    "duplicate_training_examples": 0,
    "unique_source_ids": len(set(source_ids)),
    "selection_seed": SELECTION_SEED,
    "bci_fr_extension": {
        "translated_candidates": EXTENSION_CANDIDATES,
        "selected_rows": len(selected_extension_ids),
        "unused_reserve_rows": EXTENSION_CANDIDATES - len(selected_extension_ids),
    },
    "translation_validation": translation_reports,
    "schema": list(final_rows[0].keys()),
    "language_codes": {"fr": "French", "bci": "Baoulé"},
}
write_json_atomic(FINAL_MANIFEST_PATH, manifest)
print("Dataset  :", FINAL_DATASET_PATH)
print("Manifeste:", FINAL_MANIFEST_PATH)
print("SHA-256  :", output_sha256)

Dataset  : /home/vedem_worker/scrapt-bau/data/03_final/lree_ai_fr_bci_4way_no_overlap_55k.jsonl
Manifeste: /home/vedem_worker/scrapt-bau/data/03_final/lree_ai_fr_bci_4way_no_overlap_55k_manifest.json
SHA-256  : d16020ee0daaed0be50c563ccf23c7377d037a01396637414980b1aa67f65057


## 6. Vérifier les frontières des quatre partitions

In [11]:
offset = 0
for direction in FINAL_ORDER:
    first, last = offset, offset + TARGET_COUNTS[direction] - 1
    print(f"{direction:8s}: lignes {first:,} à {last:,} | premier={final_rows[first]['id']}")
    offset = last + 1

bci_bci : lignes 0 à 16,505 | premier=french_alpaca_4a2c3b229b4999d8__bci_bci
fr_fr   : lignes 16,506 à 31,911 | premier=french_alpaca_4fb38de7a942dcf8__fr_fr
fr_bci  : lignes 31,912 à 43,465 | premier=french_alpaca_1dbab5f99118e2db__fr_bci
bci_fr  : lignes 43,466 à 55,019 | premier=french_alpaca_af1aa34587310d47__bci_fr
